In [ ]:
# Last Update: 7/4/2026 ; 1 : 11   pm
# Tsfnet optuna search

In [1]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack (kernel (3,3,3), stride (1,2,2) twice) happens to reduce
        # the 16x16 spatial grid down to 4x4 as well (verified: 16 -> 8 -> 4 with padding=1), exactly
        # matching the EEG branch's spatial resolution after its Table-1 convs. nirs_fc is updated to
        # match eeg_fc's new input size for the same reason (spatial map preserved, not pre-pooled).
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1)
        # nirs_feat = self.nirs_fc(nirs_feat_raw)
        # CHANGED: nirs_feat_raw / nirs_fc used to come from BOTH spatially- AND temporally-pooled
        # `y` (32-d, all spatial structure discarded). nirs_feat_map instead GAPs `y` over TIME ONLY
        # (mirroring eeg_feat_map above), preserving fNIRS's own (32,4,4) spatial map -- verified
        # that nirs_conv1/nirs_conv2's own strides already reduce the 16x16 grid to 4x4 (16->8->4),
        # exactly matching the EEG branch's spatial resolution. y_spatial (time-preserved,
        # spatially-pooled, used below for the existing CAFE temporal sequence) is unaffected.
        nirs_feat_map = y.mean(dim=2)  # NEW: GAP over TIME axis only -> (batch,32,4,4), spatial map preserved (paper's F^fnirs analogue)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    # criterion = nn.CrossEntropyLoss()
    # NEW (TSFNet paper Eq. 5-13 -- multi-branch + regularization loss): replacing the single
    # criterion on the final prediction alone with the full loss design from the paper. See the
    # pearson_correlation_loss() docstring and the forward()-return-value comment above for the
    # full rationale. LAMBDA_EEG and LAMBDA_FNIRS follow the paper's Eq. 6 notation (lambda_eeg,
    # lambda_fnirs).
    # CORRECTED: the comment previously here claimed "the paper... does not report their exact
    # values, so 0.3 is used" -- this was WRONG. Sec. 3.6 of the paper explicitly states: "the
    # best-performing loss weights lambda_eeg and lambda_fnirs are both set to 0.2" (found via
    # hyperparameter tuning on their own MI/MA/WG datasets). Using the paper's reported value
    # instead of an unfounded guess. L_fusion, L_efgf, and L_cafe use coefficient 1, exactly as in
    # the paper's Eq. 6 and Eq. 5 (no separate lambda given for these terms).
    criterion = nn.CrossEntropyLoss()  # NEW: still CrossEntropyLoss per-branch, just applied to 4 branches instead of 1 (see loss computation in the training loop below)
    # LAMBDA_EEG = 0.3     # NEW: Eq. 6 lambda_eeg -- thesis-specific default, paper does not report the tuned value
    # LAMBDA_FNIRS = 0.3   # NEW: Eq. 6 lambda_fnirs -- thesis-specific default, paper does not report the tuned value
    LAMBDA_EEG = 0.2     # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_FNIRS = 0.2   # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_EFGF_REG = 1.0   # NEW: Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
    LAMBDA_CAFE_REG = 1.0   # NEW: Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            # pred, _, _, _ = model(eeg_b, nirs_b)
            # 
            # loss = criterion(pred, lbl_b)
            # NEW (TSFNet paper Eq. 5-13): full multi-branch classification loss (Eq. 6) plus EFGF
            # (Eq. 12) and CAFE (Eq. 13) regularization losses, replacing the single-term loss above
            # that only supervised the final fused prediction. eeg_feat_map/efgf_attn_map are the
            # analogues used for L_efgf; nirs_feat_map/cafe_enhanced_map are the analogues used for
            # L_cafe (see forward()'s return-value comment for the exact correspondence).
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs

            L_pred = criterion(pred, lbl_b)          # NEW: Eq. 10 (generalized to 3-class CrossEntropyLoss)
            L_eeg = criterion(eeg_out, lbl_b)        # NEW: Eq. 7 (generalized to 3-class CrossEntropyLoss)
            L_fnirs = criterion(fnirs_out, lbl_b)    # NEW: Eq. 8 (generalized to 3-class CrossEntropyLoss)
            L_fusion = criterion(fusion_out, lbl_b)  # NEW: Eq. 9 (generalized to 3-class CrossEntropyLoss)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion  # NEW: Eq. 6

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)      # NEW: Eq. 12
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW: Eq. 13

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe  # NEW: Eq. 5 (total loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                # pred, _, _, _ = model(eeg_b, nirs_b)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs, only final pred needed for val accuracy
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            # pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED: forward() now returns 8 values; the 4 extra feature maps (used only for EFGF/CAFE regularization loss during training) are ignored here
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")


Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [ ]:
#without lambert wavelenght correction
#Epoch  15/100 | Train Loss: 0.7893 | Train Acc: 92.40% | Val Acc: 30.86% | Best Val: 41.92%
#fusion_out-only Test Accuracy: 35.57%

In [ ]:
#with lambert wavelenght correction
#Epoch  15/100 | Train Loss: 0.8271 | Train Acc: 91.65% | Val Acc: 37.19% | Best Val: 43.59%
#fusion_out-only Test Accuracy: 37.88%
#Overall Model Test Accuracy (final fused prediction): 37.84%

In [ ]:
#with lambert wavelenght correction+ 3fixes
#Epoch  15/100 | Train Loss: 1.1859 | Train Acc: 85.30% | Val Acc: 36.63% | Best Val: 42.53%
#Overall Model Test Accuracy (final fused prediction): 33.39%
#fusion_out-only Test Accuracy: 31.76%

In [ ]:
#Preproc sync fixed
#Epoch  15/100 | Train Loss: 0.8392 | Train Acc: 91.89% | Val Acc: 39.38% | Best Val: 42.45%
#Learned final-combination weights (after softmax): w_eeg=0.346, w_fnirs=0.321, w_fusion=0.333
#fusion_out-only Test Accuracy: 40.15%
#Overall Model Test Accuracy (final fused prediction): 38.19%

In [ ]:
#7/4/2026 9:56 pm
#Tsfnet diagnostic chronological · PY

In [2]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# NEW (diagnostic experiment -- subject-dependent split): finds contiguous "trial blocks" within a
# subject's window sequence, using label change-points as block boundaries. Windows are extracted
# trial-by-trial via a sliding window (see the preprocessing notebook's extract_synchronized_windows()),
# so consecutive windows belonging to the SAME trial sit next to each other in the saved array and
# share the same label; a label change marks the start of a new trial. This is used below to split
# each subject's OWN data into train/val/test by whole trial-blocks (not by individual window), which
# is essential: since windows overlap 66% (3s window, 1s step), a naive random per-window split would
# put near-duplicate windows from the SAME trial into both train and test, leaking information and
# producing an artificially inflated accuracy that would tell us nothing real about generalization.
def get_trial_blocks(labels):  # NEW
    """Returns a list of (start_idx, end_idx) contiguous index ranges, one per detected trial block,
    based on label change-points in the given per-window labels array."""
    blocks = []  # NEW
    start = 0  # NEW
    for i in range(1, len(labels)):  # NEW
        if labels[i] != labels[i - 1]:  # NEW
            blocks.append((start, i))  # NEW
            start = i  # NEW
    blocks.append((start, len(labels)))  # NEW
    return blocks  # NEW

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# NEW (diagnostic experiment -- Option 4, CHRONOLOGICAL variant): the first version of this
# diagnostic (SubjectDependentMMapDataset, kept elsewhere/unused here) shuffled each subject's
# trial-blocks RANDOMLY before splitting. That produced a confound noticed in the chat discussion:
# because trial order is not random in the original recording (task design), a random per-subject
# shuffle caused train/val/test to end up with very different class distributions (e.g. test's
# '3-back' class was ~38% underrepresented vs train), which alone could have depressed the reported
# accuracy independent of any real learnability question. This class instead keeps each subject's
# trial-blocks in their ORIGINAL CHRONOLOGICAL order and does a straight positional split: the
# EARLIEST ~70% of a subject's trials go to train, the next ~15% to val, and the LAST ~15%
# (chronologically most recent) to test -- no shuffling at all. This is the standard "subject-
# dependent, chronological holdout" protocol used in BCI literature (e.g. train on a subject's
# early trials, test on later ones) and removes the class-imbalance confound from the earlier
# version, at the cost of introducing a different, well-understood one: performance can also be
# affected by within-session drift (fatigue, attention, impedance changes over the recording) since
# test trials always come chronologically after train trials for every subject. This is an
# intentional, literature-standard tradeoff, not a bug -- it is flagged here for the same reason
# the earlier version's confound was flagged: so the result is interpreted correctly.
class ChronologicalSubjectDependentDataset(Dataset):  # NEW

    def __init__(self, subs_list, dataset_name, split_role, train_frac=0.70, val_frac=0.15,  # NEW
                 min_windows_threshold=500, seed=RANDOM_SEED):  # NEW
        super().__init__()  # NEW
        assert split_role in ('train', 'val', 'test')  # NEW
        self.dataset_name = dataset_name  # NEW
        self.valid_files = []  # NEW
        self.total_windows = 0  # NEW

        print(f"Scanning {dataset_name} ({split_role} blocks, CHRONOLOGICAL split) - {len(subs_list)} subjects...")  # NEW

        # per-subject block selection: figure out, for EACH subject, which contiguous trial-blocks
        # belong to train/val/test, WITHOUT loading the (large) eeg/nirs arrays yet -- only labels
        # are needed to compute block boundaries and the split.
        subject_selection = []  # NEW: list of dicts, one per subject, with the selected (start,end) ranges for this split_role
        for sub_id in subs_list:  # NEW
            sub_str = f"{sub_id:03d}"  # NEW
            subject = f"VP{sub_str}"  # NEW
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)  # NEW
            if not os.path.exists(sub_folder):  # NEW
                continue  # NEW

            eeg_file = next((f for f in [  # NEW
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"),  # NEW
                os.path.join(sub_folder, f"{subject}_eeg.npy")  # NEW
            ] if os.path.exists(f)), None)  # NEW
            nirs_file = next((f for f in [  # NEW
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"),  # NEW
                os.path.join(sub_folder, f"{subject}_fnirs.npy")  # NEW
            ] if os.path.exists(f)), None)  # NEW
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")  # NEW

            if not (eeg_file and nirs_file and os.path.exists(labels_file)):  # NEW
                continue  # NEW

            try:  # NEW
                lbls = np.load(labels_file).astype(np.int64)  # NEW
                n_windows = lbls.shape[0]  # NEW
                if n_windows < min_windows_threshold:  # NEW
                    print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}).")  # NEW
                    continue  # NEW

                blocks = get_trial_blocks(lbls)  # NEW: contiguous (start,end) trial-block ranges, in original chronological order
                # NO shuffling here (this is the key difference from SubjectDependentMMapDataset):
                # blocks stay in the exact chronological order returned by get_trial_blocks(), so
                # the positional split below directly corresponds to time -- earliest trials first.
                ordered_blocks = blocks  # NEW: alias for clarity, no copy/shuffle needed since we never mutate it

                n_blocks = len(ordered_blocks)  # NEW
                n_tr = max(1, round(n_blocks * train_frac))  # NEW
                n_va = max(1, round(n_blocks * val_frac))  # NEW
                n_tr = min(n_tr, n_blocks - 2) if n_blocks >= 3 else n_tr  # NEW: leave room for at least 1 val + 1 test block when possible
                n_va = min(n_va, max(0, n_blocks - n_tr - 1))  # NEW

                role_blocks = {  # NEW: chronological positional split -- train=earliest, val=middle, test=latest
                    'train': ordered_blocks[:n_tr],  # NEW: earliest trials
                    'val': ordered_blocks[n_tr:n_tr + n_va],  # NEW: middle trials
                    'test': ordered_blocks[n_tr + n_va:],  # NEW: latest (most recent) trials
                }[split_role]  # NEW

                if not role_blocks:  # NEW: subject contributed no blocks to this split (can happen for very short subjects); skip it for this role
                    continue  # NEW

                role_indices = np.concatenate([np.arange(s, e) for (s, e) in role_blocks])  # NEW: individual window indices selected for this split_role
                n_sel = len(role_indices)  # NEW

                subject_selection.append({  # NEW
                    'subject': subject, 'eeg_path': eeg_file, 'nirs_path': nirs_file,  # NEW
                    'labels': lbls[role_indices], 'indices': role_indices,  # NEW
                    'start_idx': self.total_windows, 'end_idx': self.total_windows + n_sel,  # NEW
                })  # NEW
                self.total_windows += n_sel  # NEW
                print(f"{subject}: {n_blocks} trial-blocks total -> {len(role_blocks)} blocks / {n_sel} windows selected for '{split_role}'")  # NEW
            except Exception as e:  # NEW
                print(f"Error scanning {subject}: {str(e)}")  # NEW
                continue  # NEW

        self.valid_files = subject_selection  # NEW
        if self.total_windows == 0:  # NEW
            raise ValueError(f"No valid preprocessed data found for {dataset_name} ({split_role}) after filtering!")  # NEW

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)  # NEW
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_ct_{dataset_name[:3]}_e.npy")  # NEW: "ct" prefix = chronological diagnostic, avoids colliding with the main pipeline's and the random-shuffle diagnostic's t_mmap_* files
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_ct_{dataset_name[:3]}_n.npy")  # NEW

        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:  # NEW
            if os.path.exists(f_path):  # NEW
                try: os.remove(f_path)  # NEW
                except: pass  # NEW
            with open(f_path, 'wb') as f:  # NEW
                pass  # NEW

        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # NEW
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))  # NEW
        self.y = np.zeros(self.total_windows, dtype='int64')  # NEW

        print(f"Streaming selected trial-blocks to disk memory-map ({dataset_name}, {split_role})...")  # NEW
        for f_info in self.valid_files:  # NEW
            s, e = f_info['start_idx'], f_info['end_idx']  # NEW
            # NEW: load the subject's FULL array once (same memory pattern as MMapHybridDataset,
            # already proven safe), then select only this split_role's window indices from it.
            full_eeg = np.load(f_info['eeg_path'])  # NEW
            full_nirs = np.load(f_info['nirs_path'])  # NEW
            self.X_eeg[s:e] = full_eeg[f_info['indices']]  # NEW
            self.X_nirs[s:e] = full_nirs[f_info['indices']]  # NEW
            self.y[s:e] = f_info['labels']  # NEW
            del full_eeg, full_nirs  # NEW
            gc.collect()  # NEW

        self.X_eeg.flush()  # NEW
        self.X_nirs.flush()  # NEW
        print(f"--> {dataset_name} ({split_role}) Ready on Disk! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")  # NEW

    def __len__(self):  # NEW
        return self.total_windows  # NEW

    def __getitem__(self, idx):  # NEW
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()  # NEW
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()  # NEW
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)  # NEW
        return eeg_tensor, nirs_tensor, label_tensor  # NEW

    def cleanup(self):  # NEW
        del self.X_eeg  # NEW
        del self.X_nirs  # NEW
        gc.collect()  # NEW
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:  # NEW
            if os.path.exists(f):  # NEW
                try: os.remove(f)  # NEW
                except: pass  # NEW

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack (kernel (3,3,3), stride (1,2,2) twice) happens to reduce
        # the 16x16 spatial grid down to 4x4 as well (verified: 16 -> 8 -> 4 with padding=1), exactly
        # matching the EEG branch's spatial resolution after its Table-1 convs. nirs_fc is updated to
        # match eeg_fc's new input size for the same reason (spatial map preserved, not pre-pooled).
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1)
        # nirs_feat = self.nirs_fc(nirs_feat_raw)
        # CHANGED: nirs_feat_raw / nirs_fc used to come from BOTH spatially- AND temporally-pooled
        # `y` (32-d, all spatial structure discarded). nirs_feat_map instead GAPs `y` over TIME ONLY
        # (mirroring eeg_feat_map above), preserving fNIRS's own (32,4,4) spatial map -- verified
        # that nirs_conv1/nirs_conv2's own strides already reduce the 16x16 grid to 4x4 (16->8->4),
        # exactly matching the EEG branch's spatial resolution. y_spatial (time-preserved,
        # spatially-pooled, used below for the existing CAFE temporal sequence) is unaffected.
        nirs_feat_map = y.mean(dim=2)  # NEW: GAP over TIME axis only -> (batch,32,4,4), spatial map preserved (paper's F^fnirs analogue)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)

# MAIN PIPELINE 
if __name__ == "__main__":
    
    # train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    # val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    # test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")
    # CHANGED (diagnostic experiment -- Option 4, CHRONOLOGICAL variant): uses ALL_SUBJECTS for
    # every split (not disjoint TRAIN/VAL/TEST subject lists), because ChronologicalSubjectDependentDataset
    # itself splits each subject's OWN trial-blocks by split_role, IN TIME ORDER (earliest ->
    # train, latest -> test) -- every subject contributes trials to all three sets, just
    # non-overlapping, chronologically ordered ones. This is NOT the thesis's main cross-subject
    # protocol; it exists only to check whether the model can learn ANYTHING when it has seen each
    # subject before, WITHOUT the class-imbalance confound of the earlier random-shuffle version.
    print("\n" + "#" * 70)  # NEW
    print("# DIAGNOSTIC RUN: SUBJECT-DEPENDENT, CHRONOLOGICAL SPLIT (Option 4, take 2)")  # NEW
    print("# NOT the thesis's main cross-subject protocol -- see chat discussion.")  # NEW
    print("# Train = each subject's EARLIEST trials, Val = middle, Test = LATEST trials.")  # NEW
    print("#" * 70 + "\n")  # NEW
    train_dataset = ChronologicalSubjectDependentDataset(ALL_SUBJECTS, "Train_Set", split_role='train')  # CHANGED
    val_dataset = ChronologicalSubjectDependentDataset(ALL_SUBJECTS, "Validation_Set", split_role='val')  # CHANGED
    test_dataset = ChronologicalSubjectDependentDataset(ALL_SUBJECTS, "Test_Set", split_role='test')  # CHANGED

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    # criterion = nn.CrossEntropyLoss()
    # NEW (TSFNet paper Eq. 5-13 -- multi-branch + regularization loss): replacing the single
    # criterion on the final prediction alone with the full loss design from the paper. See the
    # pearson_correlation_loss() docstring and the forward()-return-value comment above for the
    # full rationale. LAMBDA_EEG and LAMBDA_FNIRS follow the paper's Eq. 6 notation (lambda_eeg,
    # lambda_fnirs).
    # CORRECTED: the comment previously here claimed "the paper... does not report their exact
    # values, so 0.3 is used" -- this was WRONG. Sec. 3.6 of the paper explicitly states: "the
    # best-performing loss weights lambda_eeg and lambda_fnirs are both set to 0.2" (found via
    # hyperparameter tuning on their own MI/MA/WG datasets). Using the paper's reported value
    # instead of an unfounded guess. L_fusion, L_efgf, and L_cafe use coefficient 1, exactly as in
    # the paper's Eq. 6 and Eq. 5 (no separate lambda given for these terms).
    criterion = nn.CrossEntropyLoss()  # NEW: still CrossEntropyLoss per-branch, just applied to 4 branches instead of 1 (see loss computation in the training loop below)
    # LAMBDA_EEG = 0.3     # NEW: Eq. 6 lambda_eeg -- thesis-specific default, paper does not report the tuned value
    # LAMBDA_FNIRS = 0.3   # NEW: Eq. 6 lambda_fnirs -- thesis-specific default, paper does not report the tuned value
    LAMBDA_EEG = 0.2     # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_FNIRS = 0.2   # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_EFGF_REG = 1.0   # NEW: Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
    LAMBDA_CAFE_REG = 1.0   # NEW: Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            # pred, _, _, _ = model(eeg_b, nirs_b)
            # 
            # loss = criterion(pred, lbl_b)
            # NEW (TSFNet paper Eq. 5-13): full multi-branch classification loss (Eq. 6) plus EFGF
            # (Eq. 12) and CAFE (Eq. 13) regularization losses, replacing the single-term loss above
            # that only supervised the final fused prediction. eeg_feat_map/efgf_attn_map are the
            # analogues used for L_efgf; nirs_feat_map/cafe_enhanced_map are the analogues used for
            # L_cafe (see forward()'s return-value comment for the exact correspondence).
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs

            L_pred = criterion(pred, lbl_b)          # NEW: Eq. 10 (generalized to 3-class CrossEntropyLoss)
            L_eeg = criterion(eeg_out, lbl_b)        # NEW: Eq. 7 (generalized to 3-class CrossEntropyLoss)
            L_fnirs = criterion(fnirs_out, lbl_b)    # NEW: Eq. 8 (generalized to 3-class CrossEntropyLoss)
            L_fusion = criterion(fusion_out, lbl_b)  # NEW: Eq. 9 (generalized to 3-class CrossEntropyLoss)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion  # NEW: Eq. 6

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)      # NEW: Eq. 12
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW: Eq. 13

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe  # NEW: Eq. 5 (total loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                # pred, _, _, _ = model(eeg_b, nirs_b)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs, only final pred needed for val accuracy
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            # pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED: forward() now returns 8 values; the 4 extra feature maps (used only for EFGF/CAFE regularization loss during training) are ignored here
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")


######################################################################
# DIAGNOSTIC RUN: SUBJECT-DEPENDENT, CHRONOLOGICAL SPLIT (Option 4, take 2)
# NOT the thesis's main cross-subject protocol -- see chat discussion.
# Train = each subject's EARLIEST trials, Val = middle, Test = LATEST trials.
######################################################################

Scanning Train_Set (train blocks, CHRONOLOGICAL split) - 26 subjects...
VP001: 25 trial-blocks total -> 18 blocks / 757 windows selected for 'train'
VP002: 25 trial-blocks total -> 18 blocks / 760 windows selected for 'train'
VP003: 25 trial-blocks total -> 18 blocks / 760 windows selected for 'train'
VP004: 25 trial-blocks total -> 18 blocks / 519 windows selected for 'train'
VP005: 25 trial-blocks total -> 18 blocks / 753 windows selected for 'train'
VP006: 25 trial-blocks total -> 18 blocks / 760 windows selected for 'train'
VP007: 25 trial-blocks total -> 18 blocks / 760 windows selected for 'train'
VP008: 25 trial-bloc

In [ ]:
#without lambert wavelenght correction
#Epoch  15/100 | Train Loss: 0.7371 | Train Acc: 93.04% | Val Acc: 30.39% | Best Val: 35.44%
#Overall Model Test Accuracy (final fused prediction): 41.88%
#Learned final-combination weights (after softmax): w_eeg=0.338, w_fnirs=0.308, w_fusion=0.354
#fusion_out-only Test Accuracy: 40.34%

In [ ]:
#with lambert wavelenght correction
#Epoch  15/100 | Train Loss: 0.7901 | Train Acc: 92.27% | Val Acc: 34.55% | Best Val: 36.48%
#Overall Model Test Accuracy (final fused prediction): 37.71%
#Learned final-combination weights (after softmax): w_eeg=0.340, w_fnirs=0.309, w_fusion=0.351
#fusion_out-only Test Accuracy: 29.08%

In [ ]:
#with lambert wavelenght correction+ 3fixes
#Epoch  15/100 | Train Loss: 1.1260 | Train Acc: 85.94% | Val Acc: 32.44% | Best Val: 39.72%
#Overall Model Test Accuracy (final fused prediction): 30.89%
#Learned final-combination weights (after softmax): w_eeg=0.347, w_fnirs=0.308, w_fusion=0.345
#fusion_out-only Test Accuracy: 25.66%

In [10]:
# تعداد کاراکتر باید نزدیک به عدد اصلی فایل .py (~۲۵,۰۰۰ کاراکتر تا خط MAIN PIPELINE) باشد
print(len(model_only_code))

31469


In [2]:
import json

notebook_path = r"E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\TSFNet_model_FIXED_v6_paper_loss_new.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

full_code = ""
for cell in nb["cells"]:
    if cell["cell_type"] == "code":
        full_code += "".join(cell["source"]) + "\n\n"

# NEW: حذف خط‌هایی که دستور مخصوص Jupyter هستند (شروع با ! یا %)، چون exec() آن‌ها را نمی‌فهمد
cleaned_lines = []
for line in full_code.split("\n"):
    stripped = line.strip()
    if stripped.startswith("!") or stripped.startswith("%"):
        continue  # رد شو، این خط را در کد اجراشونده نگه نمی‌داریم
    cleaned_lines.append(line)
full_code = "\n".join(cleaned_lines)

print(f"تعداد کاراکتر کل کد (بعد از حذف دستورات Jupyter): {len(full_code)}")

boundary = full_code.index('# MAIN PIPELINE')
model_only_code = full_code[:boundary]
exec(model_only_code, globals())
print("مدل با موفقیت تعریف شد. TSFNet_With_fNIRS_2D آماده است.")

تعداد کاراکتر کل کد (بعد از حذف دستورات Jupyter): 101261
مدل با موفقیت تعریف شد. TSFNet_With_fNIRS_2D آماده است.


In [12]:
import torch

device = torch.device('cuda')
model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)
model.train()

criterion = torch.nn.CrossEntropyLoss()

def try_batch_size(bs):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        eeg = torch.randn(bs, 16, 16, 600, device=device)
        nirs = torch.randn(bs, 2, 100, 16, 16, device=device)
        lbl = torch.randint(0, 3, (bs,), device=device)

        pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_b, final_nirs_feat, enhanced = model(eeg, nirs)
        loss = criterion(pred, lbl) + criterion(eeg_out, lbl) + criterion(fnirs_out, lbl) + criterion(fusion_out, lbl)
        loss.backward()
        model.zero_grad()

        peak_gb = torch.cuda.max_memory_allocated() / 1024**3
        print(f"batch_size={bs}: OK -- peak VRAM used: {peak_gb:.2f} GB")
        return True
    except torch.cuda.OutOfMemoryError:
        print(f"batch_size={bs}: OUT OF MEMORY")
        return False

for bs in [64, 32, 16, 8]:
    if try_batch_size(bs):
        print(f"\n--> پیشنهاد: از batch_size={bs} استفاده کنید")
        break

batch_size=64: OK -- peak VRAM used: 0.32 GB

--> پیشنهاد: از batch_size=64 استفاده کنید


In [5]:
import os

folder = r"E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE"
if os.path.exists(folder):
    print("محتویات پوشه:")
    for f in os.listdir(folder):
        print(" -", f)
else:
    print("این پوشه اصلاً وجود ندارد!")

محتویات پوشه:
 - .ipynb_checkpoints
 - EEG-fNIRS_PreProc_V29_6_FIXED.ipynb
 - EEG-fNIRS_PreProc_V29_6_FIXED2 + spatial map eeg.ipynb
 - EEG-fNIRS_PreProc_V29_6_FIXED2+1616eeg+faster.ipynb
 - EEG-fNIRS_PreProc_V29_6_FIXED2.ipynb
 - EEG-fNIRS_PreProc_V29_6_FIXED3 + spatial map eeg.ipynb
 - MultiModal_ModelScript_V30.6.ipynb
 - TSFNet_model_FIXED_v6_paper_loss_new.ipynb


In [1]:
import os
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
for fname in ["t_mmap_Tra_e.npy", "t_mmap_Tra_n.npy"]:
    fpath = os.path.join(PROCESSED_DATA_PATH, fname)
    print(fname, "-> وجود دارد" if os.path.exists(fpath) else "-> پاک شده")

t_mmap_Tra_e.npy -> پاک شده
t_mmap_Tra_n.npy -> پاک شده


In [ ]:
#Tsfnet model baseline 5epochs

In [3]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack (kernel (3,3,3), stride (1,2,2) twice) happens to reduce
        # the 16x16 spatial grid down to 4x4 as well (verified: 16 -> 8 -> 4 with padding=1), exactly
        # matching the EEG branch's spatial resolution after its Table-1 convs. nirs_fc is updated to
        # match eeg_fc's new input size for the same reason (spatial map preserved, not pre-pooled).
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1)
        # nirs_feat = self.nirs_fc(nirs_feat_raw)
        # CHANGED: nirs_feat_raw / nirs_fc used to come from BOTH spatially- AND temporally-pooled
        # `y` (32-d, all spatial structure discarded). nirs_feat_map instead GAPs `y` over TIME ONLY
        # (mirroring eeg_feat_map above), preserving fNIRS's own (32,4,4) spatial map -- verified
        # that nirs_conv1/nirs_conv2's own strides already reduce the 16x16 grid to 4x4 (16->8->4),
        # exactly matching the EEG branch's spatial resolution. y_spatial (time-preserved,
        # spatially-pooled, used below for the existing CAFE temporal sequence) is unaffected.
        nirs_feat_map = y.mean(dim=2)  # NEW: GAP over TIME axis only -> (batch,32,4,4), spatial map preserved (paper's F^fnirs analogue)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    # criterion = nn.CrossEntropyLoss()
    # NEW (TSFNet paper Eq. 5-13 -- multi-branch + regularization loss): replacing the single
    # criterion on the final prediction alone with the full loss design from the paper. See the
    # pearson_correlation_loss() docstring and the forward()-return-value comment above for the
    # full rationale. LAMBDA_EEG and LAMBDA_FNIRS follow the paper's Eq. 6 notation (lambda_eeg,
    # lambda_fnirs).
    # CORRECTED: the comment previously here claimed "the paper... does not report their exact
    # values, so 0.3 is used" -- this was WRONG. Sec. 3.6 of the paper explicitly states: "the
    # best-performing loss weights lambda_eeg and lambda_fnirs are both set to 0.2" (found via
    # hyperparameter tuning on their own MI/MA/WG datasets). Using the paper's reported value
    # instead of an unfounded guess. L_fusion, L_efgf, and L_cafe use coefficient 1, exactly as in
    # the paper's Eq. 6 and Eq. 5 (no separate lambda given for these terms).
    criterion = nn.CrossEntropyLoss()  # NEW: still CrossEntropyLoss per-branch, just applied to 4 branches instead of 1 (see loss computation in the training loop below)
    # LAMBDA_EEG = 0.3     # NEW: Eq. 6 lambda_eeg -- thesis-specific default, paper does not report the tuned value
    # LAMBDA_FNIRS = 0.3   # NEW: Eq. 6 lambda_fnirs -- thesis-specific default, paper does not report the tuned value
    LAMBDA_EEG = 0.2     # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_FNIRS = 0.2   # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_EFGF_REG = 1.0   # NEW: Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
    LAMBDA_CAFE_REG = 1.0   # NEW: Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

    best_val_acc = 0.0
    # epochs = 100
    # CHANGED (baseline sanity run): reduced from 100 to 5 epochs ONLY to verify the full real-data
    # pipeline (DataLoader, forward/backward, loss computation, checkpointing) runs end-to-end
    # without errors and that train/val loss move in a sane direction, before committing to a full
    # 100-epoch run. Change back to 100 (or just re-copy from the main training file) for the real
    # training run once this baseline passes.
    epochs = 5  # CHANGED: 100 -> 5, baseline sanity run only
    print("\n" + "=" * 70)  # NEW
    print("BASELINE SANITY RUN -- only 5 epochs, NOT the real training run")  # NEW
    print("Purpose: verify the full pipeline (data loading, forward/backward, loss, checkpointing)")  # NEW
    print("runs end-to-end on real data without errors, and that loss/accuracy move sensibly.")  # NEW
    print("=" * 70 + "\n")  # NEW

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            # pred, _, _, _ = model(eeg_b, nirs_b)
            # 
            # loss = criterion(pred, lbl_b)
            # NEW (TSFNet paper Eq. 5-13): full multi-branch classification loss (Eq. 6) plus EFGF
            # (Eq. 12) and CAFE (Eq. 13) regularization losses, replacing the single-term loss above
            # that only supervised the final fused prediction. eeg_feat_map/efgf_attn_map are the
            # analogues used for L_efgf; nirs_feat_map/cafe_enhanced_map are the analogues used for
            # L_cafe (see forward()'s return-value comment for the exact correspondence).
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs

            L_pred = criterion(pred, lbl_b)          # NEW: Eq. 10 (generalized to 3-class CrossEntropyLoss)
            L_eeg = criterion(eeg_out, lbl_b)        # NEW: Eq. 7 (generalized to 3-class CrossEntropyLoss)
            L_fnirs = criterion(fnirs_out, lbl_b)    # NEW: Eq. 8 (generalized to 3-class CrossEntropyLoss)
            L_fusion = criterion(fusion_out, lbl_b)  # NEW: Eq. 9 (generalized to 3-class CrossEntropyLoss)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion  # NEW: Eq. 6

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)      # NEW: Eq. 12
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW: Eq. 13

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe  # NEW: Eq. 5 (total loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                # pred, _, _, _ = model(eeg_b, nirs_b)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs, only final pred needed for val accuracy
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            # pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED: forward() now returns 8 values; the 4 extra feature maps (used only for EFGF/CAFE regularization loss during training) are ignored here
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [ ]:
#Tsfnet diagnostic subject dependent

In [5]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# NEW (diagnostic experiment -- subject-dependent split): finds contiguous "trial blocks" within a
# subject's window sequence, using label change-points as block boundaries. Windows are extracted
# trial-by-trial via a sliding window (see the preprocessing notebook's extract_synchronized_windows()),
# so consecutive windows belonging to the SAME trial sit next to each other in the saved array and
# share the same label; a label change marks the start of a new trial. This is used below to split
# each subject's OWN data into train/val/test by whole trial-blocks (not by individual window), which
# is essential: since windows overlap 66% (3s window, 1s step), a naive random per-window split would
# put near-duplicate windows from the SAME trial into both train and test, leaking information and
# producing an artificially inflated accuracy that would tell us nothing real about generalization.
def get_trial_blocks(labels):  # NEW
    """Returns a list of (start_idx, end_idx) contiguous index ranges, one per detected trial block,
    based on label change-points in the given per-window labels array."""
    blocks = []  # NEW
    start = 0  # NEW
    for i in range(1, len(labels)):  # NEW
        if labels[i] != labels[i - 1]:  # NEW
            blocks.append((start, i))  # NEW
            start = i  # NEW
    blocks.append((start, len(labels)))  # NEW
    return blocks  # NEW

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# NEW (diagnostic experiment -- Option 4 from chat discussion): "Is there any learnable signal in
# this data at all?" The main MMapHybridDataset above splits by WHOLE SUBJECT (cross-subject
# generalization -- the hard, realistic protocol this thesis targets). This class instead splits
# EACH subject's OWN trial-blocks into train/val/test (subject-dependent flavor: every subject
# contributes to all three sets, just different trials/time-segments), which is a fundamentally
# easier task. Comparing this diagnostic's accuracy against the cross-subject run tells us whether
# poor cross-subject accuracy reflects genuine inter-subject variability (this diagnostic does well)
# or a deeper data/label problem (this diagnostic ALSO does poorly). This class is for diagnosis
# only -- it does not replace MMapHybridDataset or change the thesis's main evaluation protocol.
class SubjectDependentMMapDataset(Dataset):  # NEW

    def __init__(self, subs_list, dataset_name, split_role, train_frac=0.70, val_frac=0.15,  # NEW
                 min_windows_threshold=500, seed=RANDOM_SEED):  # NEW
        super().__init__()  # NEW
        assert split_role in ('train', 'val', 'test')  # NEW
        self.dataset_name = dataset_name  # NEW
        self.valid_files = []  # NEW
        self.total_windows = 0  # NEW

        print(f"Scanning {dataset_name} ({split_role} blocks) - {len(subs_list)} subjects...")  # NEW

        # per-subject block selection: figure out, for EACH subject, which contiguous trial-blocks
        # belong to train/val/test, WITHOUT loading the (large) eeg/nirs arrays yet -- only labels
        # are needed to compute block boundaries and the split.
        subject_selection = []  # NEW: list of dicts, one per subject, with the selected (start,end) ranges for this split_role
        for sub_id in subs_list:  # NEW
            sub_str = f"{sub_id:03d}"  # NEW
            subject = f"VP{sub_str}"  # NEW
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)  # NEW
            if not os.path.exists(sub_folder):  # NEW
                continue  # NEW

            eeg_file = next((f for f in [  # NEW
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"),  # NEW
                os.path.join(sub_folder, f"{subject}_eeg.npy")  # NEW
            ] if os.path.exists(f)), None)  # NEW
            nirs_file = next((f for f in [  # NEW
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"),  # NEW
                os.path.join(sub_folder, f"{subject}_fnirs.npy")  # NEW
            ] if os.path.exists(f)), None)  # NEW
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")  # NEW

            if not (eeg_file and nirs_file and os.path.exists(labels_file)):  # NEW
                continue  # NEW

            try:  # NEW
                lbls = np.load(labels_file).astype(np.int64)  # NEW
                n_windows = lbls.shape[0]  # NEW
                if n_windows < min_windows_threshold:  # NEW
                    print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}).")  # NEW
                    continue  # NEW

                blocks = get_trial_blocks(lbls)  # NEW: contiguous (start,end) trial-block ranges
                # NEW: deterministic per-subject shuffle -- MUST give the same partition of blocks
                # regardless of which split_role is being constructed right now (train/val/test are
                # built via 3 SEPARATE calls to this class), otherwise blocks could overlap between
                # splits. Seeding with (seed + sub_id) instead of the shared global `random` state
                # guarantees this determinism.
                block_rng = random.Random(seed + sub_id)  # NEW
                shuffled_blocks = blocks.copy()  # NEW
                block_rng.shuffle(shuffled_blocks)  # NEW

                n_blocks = len(shuffled_blocks)  # NEW
                n_tr = max(1, round(n_blocks * train_frac))  # NEW
                n_va = max(1, round(n_blocks * val_frac))  # NEW
                n_tr = min(n_tr, n_blocks - 2) if n_blocks >= 3 else n_tr  # NEW: leave room for at least 1 val + 1 test block when possible
                n_va = min(n_va, max(0, n_blocks - n_tr - 1))  # NEW

                role_blocks = {  # NEW
                    'train': shuffled_blocks[:n_tr],  # NEW
                    'val': shuffled_blocks[n_tr:n_tr + n_va],  # NEW
                    'test': shuffled_blocks[n_tr + n_va:],  # NEW
                }[split_role]  # NEW

                if not role_blocks:  # NEW: subject contributed no blocks to this split (can happen for very short subjects); skip it for this role
                    continue  # NEW

                role_indices = np.concatenate([np.arange(s, e) for (s, e) in role_blocks])  # NEW: individual window indices selected for this split_role
                n_sel = len(role_indices)  # NEW

                subject_selection.append({  # NEW
                    'subject': subject, 'eeg_path': eeg_file, 'nirs_path': nirs_file,  # NEW
                    'labels': lbls[role_indices], 'indices': role_indices,  # NEW
                    'start_idx': self.total_windows, 'end_idx': self.total_windows + n_sel,  # NEW
                })  # NEW
                self.total_windows += n_sel  # NEW
                print(f"{subject}: {n_blocks} trial-blocks total -> {len(role_blocks)} blocks / {n_sel} windows selected for '{split_role}'")  # NEW
            except Exception as e:  # NEW
                print(f"Error scanning {subject}: {str(e)}")  # NEW
                continue  # NEW

        self.valid_files = subject_selection  # NEW
        if self.total_windows == 0:  # NEW
            raise ValueError(f"No valid preprocessed data found for {dataset_name} ({split_role}) after filtering!")  # NEW

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)  # NEW
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_sd_{dataset_name[:3]}_e.npy")  # NEW: "sd" prefix = subject-dependent diagnostic, avoids colliding with the main pipeline's t_mmap_* files
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_sd_{dataset_name[:3]}_n.npy")  # NEW

        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:  # NEW
            if os.path.exists(f_path):  # NEW
                try: os.remove(f_path)  # NEW
                except: pass  # NEW
            with open(f_path, 'wb') as f:  # NEW
                pass  # NEW

        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # NEW
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))  # NEW
        self.y = np.zeros(self.total_windows, dtype='int64')  # NEW

        print(f"Streaming selected trial-blocks to disk memory-map ({dataset_name}, {split_role})...")  # NEW
        for f_info in self.valid_files:  # NEW
            s, e = f_info['start_idx'], f_info['end_idx']  # NEW
            # NEW: load the subject's FULL array once (same memory pattern as MMapHybridDataset,
            # already proven safe), then select only this split_role's window indices from it.
            full_eeg = np.load(f_info['eeg_path'])  # NEW
            full_nirs = np.load(f_info['nirs_path'])  # NEW
            self.X_eeg[s:e] = full_eeg[f_info['indices']]  # NEW
            self.X_nirs[s:e] = full_nirs[f_info['indices']]  # NEW
            self.y[s:e] = f_info['labels']  # NEW
            del full_eeg, full_nirs  # NEW
            gc.collect()  # NEW

        self.X_eeg.flush()  # NEW
        self.X_nirs.flush()  # NEW
        print(f"--> {dataset_name} ({split_role}) Ready on Disk! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")  # NEW

    def __len__(self):  # NEW
        return self.total_windows  # NEW

    def __getitem__(self, idx):  # NEW
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()  # NEW
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()  # NEW
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)  # NEW
        return eeg_tensor, nirs_tensor, label_tensor  # NEW

    def cleanup(self):  # NEW
        del self.X_eeg  # NEW
        del self.X_nirs  # NEW
        gc.collect()  # NEW
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:  # NEW
            if os.path.exists(f):  # NEW
                try: os.remove(f)  # NEW
                except: pass  # NEW

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack (kernel (3,3,3), stride (1,2,2) twice) happens to reduce
        # the 16x16 spatial grid down to 4x4 as well (verified: 16 -> 8 -> 4 with padding=1), exactly
        # matching the EEG branch's spatial resolution after its Table-1 convs. nirs_fc is updated to
        # match eeg_fc's new input size for the same reason (spatial map preserved, not pre-pooled).
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1)
        # nirs_feat = self.nirs_fc(nirs_feat_raw)
        # CHANGED: nirs_feat_raw / nirs_fc used to come from BOTH spatially- AND temporally-pooled
        # `y` (32-d, all spatial structure discarded). nirs_feat_map instead GAPs `y` over TIME ONLY
        # (mirroring eeg_feat_map above), preserving fNIRS's own (32,4,4) spatial map -- verified
        # that nirs_conv1/nirs_conv2's own strides already reduce the 16x16 grid to 4x4 (16->8->4),
        # exactly matching the EEG branch's spatial resolution. y_spatial (time-preserved,
        # spatially-pooled, used below for the existing CAFE temporal sequence) is unaffected.
        nirs_feat_map = y.mean(dim=2)  # NEW: GAP over TIME axis only -> (batch,32,4,4), spatial map preserved (paper's F^fnirs analogue)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)

# MAIN PIPELINE 
if __name__ == "__main__":
    
    # train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    # val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    # test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")
    # CHANGED (diagnostic experiment -- Option 4): uses ALL_SUBJECTS for every split (not disjoint
    # TRAIN/VAL/TEST subject lists), because SubjectDependentMMapDataset itself splits each
    # subject's OWN trial-blocks by split_role -- every subject contributes trials to all three
    # sets, just non-overlapping ones (see get_trial_blocks()/SubjectDependentMMapDataset above for
    # the leakage-prevention logic). This is NOT the thesis's main cross-subject protocol; it exists
    # only to check whether the model can learn ANYTHING when it has seen each subject before.
    print("\n" + "#" * 70)  # NEW
    print("# DIAGNOSTIC RUN: SUBJECT-DEPENDENT SPLIT (Option 4)")  # NEW
    print("# NOT the thesis's main cross-subject protocol -- see chat discussion.")  # NEW
    print("#" * 70 + "\n")  # NEW
    train_dataset = SubjectDependentMMapDataset(ALL_SUBJECTS, "Train_Set", split_role='train')  # CHANGED
    val_dataset = SubjectDependentMMapDataset(ALL_SUBJECTS, "Validation_Set", split_role='val')  # CHANGED
    test_dataset = SubjectDependentMMapDataset(ALL_SUBJECTS, "Test_Set", split_role='test')  # CHANGED

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    # criterion = nn.CrossEntropyLoss()
    # NEW (TSFNet paper Eq. 5-13 -- multi-branch + regularization loss): replacing the single
    # criterion on the final prediction alone with the full loss design from the paper. See the
    # pearson_correlation_loss() docstring and the forward()-return-value comment above for the
    # full rationale. LAMBDA_EEG and LAMBDA_FNIRS follow the paper's Eq. 6 notation (lambda_eeg,
    # lambda_fnirs).
    # CORRECTED: the comment previously here claimed "the paper... does not report their exact
    # values, so 0.3 is used" -- this was WRONG. Sec. 3.6 of the paper explicitly states: "the
    # best-performing loss weights lambda_eeg and lambda_fnirs are both set to 0.2" (found via
    # hyperparameter tuning on their own MI/MA/WG datasets). Using the paper's reported value
    # instead of an unfounded guess. L_fusion, L_efgf, and L_cafe use coefficient 1, exactly as in
    # the paper's Eq. 6 and Eq. 5 (no separate lambda given for these terms).
    criterion = nn.CrossEntropyLoss()  # NEW: still CrossEntropyLoss per-branch, just applied to 4 branches instead of 1 (see loss computation in the training loop below)
    # LAMBDA_EEG = 0.3     # NEW: Eq. 6 lambda_eeg -- thesis-specific default, paper does not report the tuned value
    # LAMBDA_FNIRS = 0.3   # NEW: Eq. 6 lambda_fnirs -- thesis-specific default, paper does not report the tuned value
    LAMBDA_EEG = 0.2     # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_FNIRS = 0.2   # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_EFGF_REG = 1.0   # NEW: Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
    LAMBDA_CAFE_REG = 1.0   # NEW: Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            # pred, _, _, _ = model(eeg_b, nirs_b)
            # 
            # loss = criterion(pred, lbl_b)
            # NEW (TSFNet paper Eq. 5-13): full multi-branch classification loss (Eq. 6) plus EFGF
            # (Eq. 12) and CAFE (Eq. 13) regularization losses, replacing the single-term loss above
            # that only supervised the final fused prediction. eeg_feat_map/efgf_attn_map are the
            # analogues used for L_efgf; nirs_feat_map/cafe_enhanced_map are the analogues used for
            # L_cafe (see forward()'s return-value comment for the exact correspondence).
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs

            L_pred = criterion(pred, lbl_b)          # NEW: Eq. 10 (generalized to 3-class CrossEntropyLoss)
            L_eeg = criterion(eeg_out, lbl_b)        # NEW: Eq. 7 (generalized to 3-class CrossEntropyLoss)
            L_fnirs = criterion(fnirs_out, lbl_b)    # NEW: Eq. 8 (generalized to 3-class CrossEntropyLoss)
            L_fusion = criterion(fusion_out, lbl_b)  # NEW: Eq. 9 (generalized to 3-class CrossEntropyLoss)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion  # NEW: Eq. 6

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)      # NEW: Eq. 12
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW: Eq. 13

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe  # NEW: Eq. 5 (total loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                # pred, _, _, _ = model(eeg_b, nirs_b)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs, only final pred needed for val accuracy
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            # pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED: forward() now returns 8 values; the 4 extra feature maps (used only for EFGF/CAFE regularization loss during training) are ignored here
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")


######################################################################
# DIAGNOSTIC RUN: SUBJECT-DEPENDENT SPLIT (Option 4)
# NOT the thesis's main cross-subject protocol -- see chat discussion.
######################################################################

Scanning Train_Set (train blocks) - 26 subjects...
VP001: 25 trial-blocks total -> 18 blocks / 760 windows selected for 'train'
VP002: 25 trial-blocks total -> 18 blocks / 722 windows selected for 'train'
VP003: 25 trial-blocks total -> 18 blocks / 722 windows selected for 'train'
VP004: 25 trial-blocks total -> 18 blocks / 489 windows selected for 'train'
VP005: 25 trial-blocks total -> 18 blocks / 750 windows selected for 'train'
VP006: 25 trial-blocks total -> 18 blocks / 722 windows selected for 'train'
VP007: 25 trial-blocks total -> 18 blocks / 722 windows selected for 'train'
VP008: 25 trial-blocks total -> 18 blocks / 719 windows selected for 'train'
VP009: 25 trial-blocks total -> 18 blocks / 720 windows selected 